Load the Datasets 

In [5]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw/NHANES_2017_2018")

demo  = pd.read_sas(DATA_DIR / "DEMO_J.xpt")
bmx   = pd.read_sas(DATA_DIR / "BMX_J.xpt")
bpx   = pd.read_sas(DATA_DIR / "BPX_J.xpt")
glu   = pd.read_sas(DATA_DIR / "GLU_J.xpt")
ghb   = pd.read_sas(DATA_DIR / "GHB_J.xpt")
ins   = pd.read_sas(DATA_DIR / "INS_J.xpt")
tchol = pd.read_sas(DATA_DIR / "TCHOL_J.xpt")
hdl   = pd.read_sas(DATA_DIR / "HDL_J.xpt")

print("All NHANES tables loaded")


All NHANES tables loaded


In [6]:
demo.columns


Index(['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN',
       'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC',
       'DMDBORN4', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDMARTL',
       'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY',
       'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'DMDHHSIZ',
       'DMDFMSIZ', 'DMDHHSZA', 'DMDHHSZB', 'DMDHHSZE', 'DMDHRGND', 'DMDHRAGZ',
       'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU',
       'SDMVSTRA', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR'],
      dtype='object')

Selecting columns from each table

Demographics Age And Gender

In [7]:
demo_sel = (
    demo[["SEQN", "RIDAGEYR", "RIAGENDR"]]
    .copy()
    .rename(columns={
        "RIDAGEYR": "Age",
        "RIAGENDR": "Gender"
    })
)


BMI

In [8]:
bmx_sel = (
    bmx[["SEQN", "BMXBMI"]]
    .copy()
    .rename(columns={"BMXBMI": "BMI"})
)


Average of Blood Pressure

In [9]:
bpx_temp = bpx[[
    "SEQN",
    "BPXSY1", "BPXSY2", "BPXSY3",
    "BPXDI1", "BPXDI2", "BPXDI3"
]].copy()

bpx_temp["Systolic_BP"] = bpx_temp[["BPXSY1", "BPXSY2", "BPXSY3"]].mean(axis=1)
bpx_temp["Diastolic_BP"] = bpx_temp[["BPXDI1", "BPXDI2", "BPXDI3"]].mean(axis=1)

bpx_sel = bpx_temp[["SEQN", "Systolic_BP", "Diastolic_BP"]]


Glucose

In [10]:
glu_sel = (
    glu[["SEQN", "LBXGLU"]]
    .copy()
    .rename(columns={"LBXGLU": "Glucose"})
)


OUR TARGET

In [11]:
ghb_sel = (
    ghb[["SEQN", "LBXGH"]]
    .copy()
    .rename(columns={"LBXGH": "HbA1c"})
)


Insulin

In [12]:
ins_sel = (
    ins[["SEQN", "LBXIN"]]
    .copy()
    .rename(columns={"LBXIN": "Insulin"})
)


In [13]:
ins.columns


Index(['SEQN', 'WTSAF2YR', 'LBXIN', 'LBDINSI', 'LBDINLC'], dtype='object')

Cholestrol

In [14]:
tchol_sel = (
    tchol[["SEQN", "LBXTC"]]
    .copy()
    .rename(columns={"LBXTC": "Total_Cholesterol"})
)

hdl_sel = (
    hdl[["SEQN", "LBDHDD"]]
    .copy()
    .rename(columns={"LBDHDD": "HDL_Cholesterol"})
)


Merging into ONE Patient State table 

In [15]:
from functools import reduce

dfs = [
    demo_sel,
    bmx_sel,
    bpx_sel,
    glu_sel,
    ghb_sel,
    ins_sel,
    tchol_sel,
    hdl_sel
]

patient_state = reduce(
    lambda left, right: pd.merge(left, right, on="SEQN", how="inner"),
    dfs
)


In [16]:
patient_state.head()


,SEQN,Age,Gender,BMI,Systolic_BP,Diastolic_BP,Glucose,HbA1c,Insulin,Total_Cholesterol,HDL_Cholesterol
0,93708.0,66.0,2.0,23.7,141.000000,77.000000,122.0,6.2,9.72,209.0,88.0
1,93711.0,56.0,1.0,21.3,101.333333,66.666667,107.0,5.7,5.28,238.0,72.0
2,93717.0,22.0,1.0,24.5,118.666667,65.333333,91.0,5.1,3.94,213.0,53.0
3,93718.0,45.0,1.0,22.0,131.333333,90.000000,89.0,5.7,4.89,152.0,63.0
4,93719.0,13.0,2.0,26.0,101.333333,64.000000,86.0,5.0,10.94,97.0,46.0


In [17]:
patient_state.shape


(3036, 11)

In [18]:
patient_state.isna().sum()


SEQN                   0
Age                    0
Gender                 0
BMI                   55
Systolic_BP          165
Diastolic_BP         165
Glucose              145
HbA1c                137
Insulin              211
Total_Cholesterol    196
HDL_Cholesterol      196
dtype: int64

Encode Gender

In [19]:
patient_state["Gender"] = patient_state["Gender"].map({1.0: 0, 2.0: 1})


Handle missing values

In [20]:
from sklearn.impute import SimpleImputer

features = [
    "Age", "Gender", "BMI",
    "Systolic_BP", "Diastolic_BP",
    "Glucose", "Insulin",
    "Total_Cholesterol", "HDL_Cholesterol"
]

imputer = SimpleImputer(strategy="median")
patient_state[features] = imputer.fit_transform(patient_state[features])


Verify missing values

In [21]:
patient_state.isna().sum()


SEQN                   0
Age                    0
Gender                 0
BMI                    0
Systolic_BP            0
Diastolic_BP           0
Glucose                0
HbA1c                137
Insulin                0
Total_Cholesterol      0
HDL_Cholesterol        0
dtype: int64

In [22]:
# Drop missing HbA1c BEFORE labeling
patient_state = patient_state.dropna(subset=["HbA1c"])

# Create metabolic risk label
patient_state["Metabolic_Risk"] = (patient_state["HbA1c"] >= 5.7).astype(int)

# Save again
patient_state.to_csv(
    "../data/processed/patient_state_clean.csv",
    index=False
)

In [23]:
patient_state.to_csv(
    "../data/processed/patient_state_clean.csv",
    index=False
)


In [24]:
# ==============================
# DATASET SUMMARY FOR PAPER
# ==============================

import pandas as pd

data = pd.read_csv("../data/processed/patient_state_clean.csv")

print("Final Dataset Shape:", data.shape)

print("\nClass Distribution:")
print(data["Metabolic_Risk"].value_counts())

print("\nClass Percentage:")
print(data["Metabolic_Risk"].value_counts(normalize=True) * 100)

print("\nDescriptive Statistics:")
print(data.describe())

print("\nMissing Values:")
print(data.isna().sum())

Final Dataset Shape: (2899, 12)

Class Distribution:
Metabolic_Risk
0    1731
1    1168
Name: count, dtype: int64

Class Percentage:
Metabolic_Risk
0    59.710245
1    40.289755
Name: proportion, dtype: float64

Descriptive Statistics:
                SEQN          Age       Gender          BMI  Systolic_BP  \
count    2899.000000  2899.000000  2899.000000  2899.000000  2899.000000   
mean    98356.438772    45.783029     0.519489    28.955157   123.758997   
std      2693.024120    20.874833     0.499706     7.451032    19.721295   
min     93708.000000    12.000000     0.000000    14.600000    74.000000   
25%     96046.000000    27.000000     0.000000    23.900000   110.000000   
50%     98322.000000    47.000000     1.000000    27.800000   120.000000   
75%    100711.500000    63.000000     1.000000    32.800000   133.333333   
max    102956.000000    80.000000     1.000000    86.200000   224.666667   

       Diastolic_BP      Glucose        HbA1c      Insulin  Total_Cholesterol  

In [25]:
import pandas as pd

data = pd.read_csv("../data/processed/patient_state_clean.csv")

print("Final Dataset Shape:", data.shape)

print("\nClass Distribution:")
print(data["Metabolic_Risk"].value_counts())

print("\nClass Percentage:")
print(data["Metabolic_Risk"].value_counts(normalize=True) * 100)

print("\nMissing Values:")
print(data.isna().sum())

Final Dataset Shape: (2899, 12)

Class Distribution:
Metabolic_Risk
0    1731
1    1168
Name: count, dtype: int64

Class Percentage:
Metabolic_Risk
0    59.710245
1    40.289755
Name: proportion, dtype: float64

Missing Values:
SEQN                 0
Age                  0
Gender               0
BMI                  0
Systolic_BP          0
Diastolic_BP         0
Glucose              0
HbA1c                0
Insulin              0
Total_Cholesterol    0
HDL_Cholesterol      0
Metabolic_Risk       0
dtype: int64
